# SAM-Body4D 推論ノートブック（00：動画→22関節JSON）

**目標：** 自分の動画（または配布サンプル動画）を SAM-Body4D に通し、実習ノートブック
（本編の実習ノート`01_handson.ipynb`）と同じ形式の22関節JSONを生成する。

## はじめに：このノートを動かす前に

このノートは、本編・発展のノート（MediaPipe）と違い、使う前に準備が要ります。準備の手順は「事前準備の手順書」（docs/handout-hf-setup.md）にあります。

- Hugging Face（ハギングフェイス）：AIのモデル（学習済みの重み）を配るサイトです。アカウントは無料で作れます。
- 利用申請：このノートが使う2つのモデル（facebook/sam3、facebook/sam-3d-body-dinov3）は、利用規約に同意して申請し、承認されてから使えます。承認には数日かかることがあるので、早めに申請してください。
- アクセストークン：承認されたら作る「合言葉」（hf_で始まる文字列）です。下の`login()`のセルで貼ります。人に見せないでください。
- ダウンロード：重みは合計約24GBで、取ってくるだけで20〜30分かかります。Colabは閉じると中身が消えるので、使うたびに取り直しになります。

## 所要時間の目安
- セットアップ〜チェックポイント取得（初回のみ）: **20〜30分**（合計約24GBのダウンロード）
- 動画の前処理〜推論〜JSON変換: 動画の長さ・GPU混雑状況にもよるが数分程度

## 実行前に必ず確認すること
1. **ランタイム → ランタイムのタイプを変更 → GPU（T4）を選択してから実行すること。**
   T4以外（CPU/A100/L4等）では動作しないか極端に遅くなる。
2. **HuggingFace の事前準備が完了していること。**
   `facebook/sam3` と `facebook/sam-3d-body-dinov3` の**両方**のゲート付きモデルページで
   利用申請の承認を得ておく必要がある（配布フォルダ `00_事前準備/` の手順書を参照）。
   未承認のままだとチェックポイント取得セルで `403 Forbidden` になる。
3. 各セルは**上から順に**実行すること（Shift+Enter）。

## このノートの使い方
- セル2: GPU確認（T4でなければ警告）
- セル3〜6: SAM-Body4D のセットアップ・HFログイン・チェックポイント取得・設定変更
- セル7〜9: 動画の入力・推論実行・JSON変換（ダウンロード可能）
- セル10: 結果の簡易確認（肩挙上角の時系列・レンダリング画像）
- セル11: トラブルシューティング


In [ ]:
!nvidia-smi

import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True,
)
gpu_name = result.stdout.strip()
print(f'検出したGPU: {gpu_name or "（検出できませんでした）"}')

if 'T4' not in gpu_name:
    print('\n⚠️ 警告: T4 GPU が検出されませんでした。')
    print('  ランタイム → ランタイムのタイプを変更 → T4 GPU を選択してから、')
    print('  このノートブックを最初から再実行してください。')
else:
    print('✅ T4 GPU を検出しました。このまま進めます。')


In [ ]:
# torch は Colab プリインストール版をそのまま使う（下記コマンドで torch 自体は再インストールしない）。
# もし後続セルで「torch 2.7.1 が必要」等のバージョン不整合エラーが出た場合のみ、
# 次の行のコメントを外して pinned install を実行し、その後このセル以降を再実行すること:
# !pip -q install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126

!git clone https://github.com/gaomingqi/sam-body4d.git
%cd sam-body4d

# detectron2 のビルドには Python.h（CPython開発ヘッダ）が必要。
# Colab の Python には通常 Python.h が含まれるため成功する見込みだが、
# 万一 "Python.h: No such file or directory" 等で失敗した場合は
# セル11のトラブルシューティングを参照し、ランタイムを再起動してから再実行すること。
!pip -q install 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps
!pip -q install -e models/sam3
!pip -q install -e .

print('セットアップ完了')


In [ ]:
# 実行するとトークン入力用のUIが表示される。
# facebook/sam3 と facebook/sam-3d-body-dinov3 の両方でゲート承認済みのアカウントの
# トークン（Read権限で可）を貼り付けること。未承認の場合は次のチェックポイント取得セルで
# 403 Forbidden になる（セル11参照）。
from huggingface_hub import login

login()


In [ ]:
# 合計約24GB（sam3, sam-3d-body-dinov3, moge-2, diffusion-vas×2, depth-anything-v2）を取得する。
# completion（Diffusion-VAS）は次セルでOFFにするが、setup.py の仕様上ダウンロード自体はまとめて
# 行われるため省略はできない。初回は回線状況にもよるが目安 20〜30分。
!python scripts/setup.py --ckpt-root /content/ckpts


In [ ]:
from omegaconf import OmegaConf

CFG_PATH = 'configs/body4d.yaml'

cfg = OmegaConf.load(CFG_PATH)
cfg.completion.enable = False       # Diffusion-VAS の遮蔽補完はT4のVRAMでは不足するためOFF
cfg.sam_3d_body.batch_size = 8      # T4(16GB)向けにバッチを縮小（サーバーA6000ではデフォルト64）
OmegaConf.save(cfg, CFG_PATH)

print('body4d.yaml を更新しました: completion.enable=False, sam_3d_body.batch_size=8')


In [ ]:
import os

from google.colab import files

USE_SAMPLE_VIDEO = True  # True：配布の正面の動画（REHAB24-6、正解つき）を使う。False：自分の動画をアップロードする
SAMPLE_VIDEO_URL = 'https://k0heiun0.github.io/pt_seminar/data/rehab24/front.mp4'

if USE_SAMPLE_VIDEO:
    !wget -q -O input_raw.mp4 {SAMPLE_VIDEO_URL}
    print('サンプル動画をダウンロードしました: input_raw.mp4')
else:
    uploaded = files.upload()  # 自分の動画ファイルを選択
    raw_name = next(iter(uploaded.keys()))
    os.rename(raw_name, 'input_raw.mp4')
    print(f'アップロード完了: {raw_name} → input_raw.mp4')

# GPUメモリ・処理時間を抑えるため、最大10秒・幅720pxに縮小する
!ffmpeg -y -i input_raw.mp4 -t 10 -vf "scale=720:-2" -an input_video.mp4
print('前処理済み動画: input_video.mp4')


In [ ]:
# --- scripts/body4d_run_video.py の dump_keypoints と同期すること ---
import argparse
import os
import sys
from pathlib import Path

import numpy as np


def dump_keypoints(outputs, id_current, image_path, dump_root):
    if not outputs:
        return
    stem = Path(image_path).stem
    ids = id_current if id_current is not None else range(1, len(outputs) + 1)
    for obj_id, person in zip(ids, outputs):
        out_dir = Path(dump_root) / "keypoints_4d" / str(obj_id)
        out_dir.mkdir(parents=True, exist_ok=True)
        np.savez(
            out_dir / f"{stem}.npz",
            keypoints_3d=np.asarray(person["pred_keypoints_3d"], dtype=np.float32),
            focal_length=np.float32(np.asarray(person["focal_length"]).reshape(-1)[0]),
            cam_t=np.asarray(person["pred_cam_t"], dtype=np.float32).reshape(-1),
        )


# sam-body4d/scripts 内の offline_app.py を import できるようにする
sys.path.insert(0, 'scripts')
import offline_app  # noqa: E402

OUTPUT_DIR = '/content/out'

_orig_save_mesh_results = offline_app.save_mesh_results


def _wrapped_save_mesh_results(*, outputs, faces, save_dir, focal_dir, image_path, id_current):
    dump_keypoints(outputs, id_current, image_path, OUTPUT_DIR)
    return _orig_save_mesh_results(
        outputs=outputs, faces=faces, save_dir=save_dir,
        focal_dir=focal_dir, image_path=image_path, id_current=id_current,
    )


offline_app.save_mesh_results = _wrapped_save_mesh_results

os.makedirs(OUTPUT_DIR, exist_ok=True)
args = argparse.Namespace(input_video='input_video.mp4', output_dir=OUTPUT_DIR)
offline_app.inference(args)

print('推論完了:', OUTPUT_DIR)


In [ ]:
# --- scripts/sam3d_body/mhr22.py, scripts/preprocess_body4d.py と同期すること ---
import json
import subprocess
from pathlib import Path

import numpy as np

MHR70_NAMES = [
    "nose", "left-eye", "right-eye", "left-ear", "right-ear",
    "left-shoulder", "right-shoulder", "left-elbow", "right-elbow",
    "left-hip", "right-hip", "left-knee", "right-knee", "left-ankle", "right-ankle",
    "left-big-toe-tip", "left-small-toe-tip", "left-heel",
    "right-big-toe-tip", "right-small-toe-tip", "right-heel",
    "right-thumb-tip", "right-thumb-first-joint", "right-thumb-second-joint",
    "right-thumb-third-joint", "right-index-tip", "right-index-first-joint",
    "right-index-second-joint", "right-index-third-joint", "right-middle-tip",
    "right-middle-first-joint", "right-middle-second-joint", "right-middle-third-joint",
    "right-ring-tip", "right-ring-first-joint", "right-ring-second-joint",
    "right-ring-third-joint", "right-pinky-tip", "right-pinky-first-joint",
    "right-pinky-second-joint", "right-pinky-third-joint", "right-wrist",
    "left-thumb-tip", "left-thumb-first-joint", "left-thumb-second-joint",
    "left-thumb-third-joint", "left-index-tip", "left-index-first-joint",
    "left-index-second-joint", "left-index-third-joint", "left-middle-tip",
    "left-middle-first-joint", "left-middle-second-joint", "left-middle-third-joint",
    "left-ring-tip", "left-ring-first-joint", "left-ring-second-joint",
    "left-ring-third-joint", "left-pinky-tip", "left-pinky-first-joint",
    "left-pinky-second-joint", "left-pinky-third-joint", "left-wrist",
    "left-olecranon", "right-olecranon", "left-cubital-fossa", "right-cubital-fossa",
    "left-acromion", "right-acromion", "neck",
]
_IDX = {n: i for i, n in enumerate(MHR70_NAMES)}
SMPL22_NAMES = [
    "pelvis", "left_hip", "right_hip", "spine1", "left_knee", "right_knee",
    "spine2", "left_ankle", "right_ankle", "spine3", "left_foot", "right_foot",
    "neck", "left_collar", "right_collar", "head",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
]


def mhr70_to_smpl22(kp70):
    P = np.asarray(kp70, dtype=np.float64).reshape(70, 3)
    def g(name): return P[_IDX[name]]
    pelvis = (g("left-hip") + g("right-hip")) / 2.0
    neck = g("neck")
    def spine(alpha): return pelvis + alpha * (neck - pelvis)
    return np.stack([
        pelvis, g("left-hip"), g("right-hip"), spine(0.25),
        g("left-knee"), g("right-knee"), spine(0.50),
        g("left-ankle"), g("right-ankle"), spine(0.75),
        (g("left-big-toe-tip") + g("left-heel")) / 2.0,
        (g("right-big-toe-tip") + g("right-heel")) / 2.0,
        neck, (neck + g("left-shoulder")) / 2.0, (neck + g("right-shoulder")) / 2.0,
        (g("left-ear") + g("right-ear")) / 2.0,
        g("left-shoulder"), g("right-shoulder"),
        g("left-elbow"), g("right-elbow"), g("left-wrist"), g("right-wrist"),
    ])


AXIS_FLIP = np.array([1.0, -1.0, -1.0])


def convert_frames(frames, fps, source_video):
    joints = np.stack([mhr70_to_smpl22(f) * AXIS_FLIP for f in frames])
    vel_left = np.diff(joints[:, 20, :], axis=0) * fps
    vel_right = np.diff(joints[:, 21, :], axis=0) * fps
    return {
        "meta": {
            "source_video": source_video, "fps": float(fps),
            "n_frames": int(joints.shape[0]),
            "joint_names": {str(i): n for i, n in enumerate(SMPL22_NAMES)},
        },
        "joints": joints.tolist(),
        "wrist_speed_left": np.linalg.norm(vel_left, axis=1).tolist(),
        "wrist_speed_right": np.linalg.norm(vel_right, axis=1).tolist(),
    }


def load_keypoints_dir(kp_dir):
    """npz ディレクトリを読み、フレーム番号順の (70,3) リストを返す。

    フレーム番号はファイル名（%08d.npz）から取り、抜けは前方補完する。
    先頭抜けは最初の検出フレームで後方補完する。
    """
    files = sorted(Path(kp_dir).glob("*.npz"))
    if not files:
        raise FileNotFoundError(f"npz が見つかりません: {kp_dir}")
    by_index = {int(f.stem): np.asarray(np.load(f)["keypoints_3d"]) for f in files}
    first, last = min(by_index), max(by_index)
    frames = []
    prev = by_index[first]
    for t in range(0, last + 1):
        cur = by_index.get(t, prev)
        frames.append(cur)
        prev = cur
    return frames


# --- 実行: input_video.mp4 のfpsを取得し、keypoints_4d/1 を22関節JSONに変換 ---
fps_str = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v', '-show_entries',
     'stream=r_frame_rate', '-of', 'default=noprint_wrappers=1:nokey=1', 'input_video.mp4'],
    capture_output=True, text=True,
).stdout.strip()
_num, _den = fps_str.split('/')
fps = float(_num) / float(_den)

KEYPOINTS_DIR = f'{OUTPUT_DIR}/keypoints_4d/1'  # 1人目（obj_id=1）を使用
frames = load_keypoints_dir(KEYPOINTS_DIR)
data = convert_frames(frames, fps=fps, source_video='input_video.mp4')

with open('my_output.json', 'w') as f:
    json.dump(data, f)

print(f"フレーム数: {data['meta']['n_frames']}, fps: {fps:.2f}")
print('保存しました: my_output.json（01_handson.ipynb のデータと同形式）')

from google.colab import files as _files
_files.download('my_output.json')


In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt


def _angle_between(v1, v2):
    v1n = v1 / (np.linalg.norm(v1) + 1e-8)
    v2n = v2 / (np.linalg.norm(v2) + 1e-8)
    return np.degrees(np.arccos(np.clip(np.dot(v1n, v2n), -1.0, 1.0)))


def _shoulder_elevation(joints_frame, side='right'):
    # SMPL22_NAMES: 16=left_shoulder, 17=right_shoulder, 18=left_elbow, 19=right_elbow,
    #               0=pelvis, 12=neck
    shoulder, elbow = (joints_frame[17], joints_frame[19]) if side == 'right' \
        else (joints_frame[16], joints_frame[18])
    pelvis, neck = joints_frame[0], joints_frame[12]
    return _angle_between(elbow - shoulder, pelvis - neck)


joints = np.array(data['joints'])  # (T, 22, 3)
n_frames = data['meta']['n_frames']
time = np.arange(n_frames) / fps
side = 'right'
elevation = np.array([_shoulder_elevation(joints[i], side) for i in range(n_frames)])

plt.figure(figsize=(10, 4))
plt.plot(time, elevation, color='#0f3460', linewidth=2)
plt.xlabel('時間 (秒)')
plt.ylabel('肩挙上角 (°)')
plt.title(f'肩挙上角の時系列（{side}側・簡易確認用）')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# rendered_frames から開始・中間・終了付近の3枚を表示（メッシュ重畳の目視確認用）
rendered_dir = Path(OUTPUT_DIR) / 'rendered_frames'
rendered_files = sorted(rendered_dir.glob('*.jpg')) or sorted(rendered_dir.glob('*.png'))

if rendered_files:
    picks = [rendered_files[0], rendered_files[len(rendered_files) // 2], rendered_files[-1]]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, p in zip(axes, picks):
        ax.imshow(mpimg.imread(p))
        ax.set_title(p.name)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'rendered_frames が見つかりませんでした（{rendered_dir} を確認してください）')


## 正解と比べる（配布の動画を使ったときだけ）

配布の正面の動画（REHAB24-6）には、モーションキャプチャーの正解があります。SAM-Body4Dの肩挙上角を、本編と同じ定義（上腕と、両肩の中点→両股の中点の線のなす角）で出し、正解と比べます。本編のMediaPipeでは、正面の動画の正解との差は2Dで4.3°、3Dで4.4°でした（29.3秒全体）。

このノートは、講師の手元（GPUなし）では最後まで確かめていません。Colab（GPU）で動かしたときの結果を教えてもらえると助かります。

In [ ]:
if USE_SAMPLE_VIDEO:
    !wget -q https://k0heiun0.github.io/pt_seminar/data/rehab24/front_truth.csv
    truth = np.genfromtxt('front_truth.csv', delimiter=',', skip_header=1, usecols=2)

    def elevation_mid(j):
        # SMPL22：16 左肩、17 右肩、19 右肘、1 左股、2 右股（本編と同じ定義にそろえる）
        mid_shoulder = (j[16] + j[17]) / 2
        mid_hip = (j[1] + j[2]) / 2
        return _angle_between(j[19] - j[17], mid_hip - mid_shoulder)

    sam = np.array([elevation_mid(joints[i]) for i in range(n_frames)])
    n = min(len(sam), len(truth))     # 前処理で最初の10秒に切っている
    print('比べたコマ数', n)
    print('SAM-Body4Dと正解の差', round(np.nanmean(np.abs(sam[:n] - truth[:n])), 1))
    t = np.arange(n) / fps
    plt.plot(t, truth[:n], color='black', label='正解')
    plt.plot(t, sam[:n], label='SAM-Body4D')
    plt.xlabel('時間 [秒]')
    plt.ylabel('肩挙上角 [度]')
    plt.legend()
    plt.show()

## トラブルシューティング

**OOM（CUDA out of memory）が出る**
- 動画をさらに短く（例: 5秒）・小さく（例: 幅480px）してセル7からやり直す。
- セル6の `sam_3d_body.batch_size` を 8 → 4 などさらに小さくして再実行する。
- それでも解消しない場合は、講師のデモを見るか、本編・発展のノート（MediaPipe、GPU不要）で続けてよい（このノートは任意）。

**HuggingFace で 403 Forbidden になる（セル5のチェックポイント取得）**
- `facebook/sam3` と `facebook/sam-3d-body-dinov3` の**どちらか、または両方**の
  利用申請が未承認・未提出の可能性が高い。各モデルページで申請状況を確認し、
  承認が下りてからセル4（HFログイン）からやり直す。
- セル4で貼り付けたトークンが正しいアカウント（申請したアカウント）のものか確認する。

**detectron2 のビルドに失敗する（セル3、"Python.h: No such file or directory" 等）**
- **ランタイム → ランタイムを再起動**してから、セル2から再実行する。
- それでも失敗する場合はセル3冒頭のコメントに従い、pinned torch install
  （torch==2.7.1 系）を試してから再実行する。

**ランタイムが切断された（長時間の推論中など）**
- ランタイムに再接続し、セル2（GPU確認）から順に再実行する。
  `git clone` 済みのリポジトリやダウンロード済みチェックポイントはランタイムの
  ディスクごと消えている場合があるため、セル3・セル5からやり直しになることが多い。
- 何度も切断される場合は動画を短くして再試行するか、講師デモ＋配布JSONに切り替える。
